# 🚀 Python Classes - Advanced Concepts Practice Guide

This notebook covers **advanced** Python OOP concepts with examples and exercises.

## Table of Contents
1. Abstract Base Classes (ABC)
2. Dataclasses
3. `__slots__` for Memory Optimization
4. Descriptors
5. Metaclasses
6. Context Managers with Classes
7. Mixins
8. Method Resolution Order (MRO) Deep Dive
9. Class Decorators
10. Singleton Pattern

---
## 1. Abstract Base Classes (ABC)

**Overview:** ABCs define a blueprint for other classes. They can have abstract methods that MUST be implemented by subclasses.

**When to Use:**
- Enforce a contract/interface
- Ensure subclasses implement specific methods
- Design frameworks and APIs

**Workflow:**
```python
from abc import ABC, abstractmethod

class AbstractClass(ABC):
    @abstractmethod
    def must_implement(self):
        pass
    
    def concrete_method(self):
        return "This is implemented"
```

In [ ]:
# Example: Abstract Base Class
from abc import ABC, abstractmethod

class Shape(ABC):
    """Abstract base class for shapes"""
    
    @abstractmethod
    def area(self):
        """Calculate area - must be implemented"""
        pass
    
    @abstractmethod
    def perimeter(self):
        """Calculate perimeter - must be implemented"""
        pass
    
    def description(self):
        """Concrete method - inherited as-is"""
        return f"I am a {self.__class__.__name__}"

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height
    
    def area(self):
        return self.width * self.height
    
    def perimeter(self):
        return 2 * (self.width + self.height)

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    
    def area(self):
        import math
        return math.pi * self.radius ** 2
    
    def perimeter(self):
        import math
        return 2 * math.pi * self.radius

# Cannot instantiate ABC directly
# shape = Shape()  # TypeError!

rect = Rectangle(5, 3)
circle = Circle(4)

print(f"{rect.description()}: Area={rect.area()}, Perimeter={rect.perimeter()}")
print(f"{circle.description()}: Area={circle.area():.2f}, Perimeter={circle.perimeter():.2f}")

In [ ]:
# ✏️ Exercise 1: Create an abstract class PaymentProcessor with:
# - Abstract methods: process_payment(amount), refund(amount)
# - Concrete method: validate_amount(amount) that checks amount > 0
# Create two implementations: CreditCardProcessor and PayPalProcessor

# Your code here:


---
## 2. Dataclasses

**Overview:** `@dataclass` decorator automatically generates `__init__`, `__repr__`, `__eq__`, and more.

**When to Use:**
- Classes primarily storing data
- Reduce boilerplate code
- Need automatic comparison/hashing

**Workflow:**
```python
from dataclasses import dataclass, field

@dataclass
class MyClass:
    name: str
    value: int = 0  # Default value
    items: list = field(default_factory=list)  # Mutable default
```

In [ ]:
# Example: Dataclasses
from dataclasses import dataclass, field
from typing import List

@dataclass
class Product:
    name: str
    price: float
    quantity: int = 0
    tags: List[str] = field(default_factory=list)
    
    def total_value(self):
        return self.price * self.quantity

@dataclass(frozen=True)  # Immutable
class Point:
    x: float
    y: float

@dataclass(order=True)  # Enables comparison operators
class Student:
    sort_index: float = field(init=False, repr=False)
    name: str
    grade: float
    
    def __post_init__(self):
        self.sort_index = self.grade

# Auto-generated __init__, __repr__, __eq__
p1 = Product("Laptop", 999.99, 5, ["electronics", "computers"])
p2 = Product("Laptop", 999.99, 5, ["electronics", "computers"])

print(f"Product: {p1}")
print(f"p1 == p2: {p1 == p2}")
print(f"Total value: ${p1.total_value()}")

# Frozen dataclass
point = Point(3.0, 4.0)
print(f"\nPoint: {point}")
# point.x = 5  # FrozenInstanceError!

# Ordered dataclass
students = [Student("Bob", 85), Student("Alice", 92), Student("Charlie", 78)]
print(f"\nSorted students: {sorted(students)}")

In [ ]:
# ✏️ Exercise 2: Create a dataclass called Order with:
# - order_id: str
# - customer: str
# - items: List[str] (default empty list)
# - total: float (default 0.0)
# - Add a method add_item(item, price) that appends item and updates total

# Your code here:


---
## 3. `__slots__` for Memory Optimization

**Overview:** `__slots__` restricts attributes and reduces memory by avoiding `__dict__`.

**When to Use:**
- Creating millions of instances
- Memory optimization is critical
- Fixed set of attributes

**Workflow:**
```python
class OptimizedClass:
    __slots__ = ['attr1', 'attr2']
    
    def __init__(self, attr1, attr2):
        self.attr1 = attr1
        self.attr2 = attr2
```

In [ ]:
# Example: __slots__
import sys

class RegularPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class SlottedPoint:
    __slots__ = ['x', 'y']
    
    def __init__(self, x, y):
        self.x = x
        self.y = y

# Compare memory usage
regular = RegularPoint(1, 2)
slotted = SlottedPoint(1, 2)

print(f"Regular Point has __dict__: {hasattr(regular, '__dict__')}")
print(f"Slotted Point has __dict__: {hasattr(slotted, '__dict__')}")

print(f"\nRegular Point dict: {regular.__dict__}")
# print(slotted.__dict__)  # AttributeError!

# Cannot add new attributes to slotted class
regular.z = 3  # Works fine
# slotted.z = 3  # AttributeError!

print(f"\nAdded z to regular: {regular.z}")

# Memory comparison (approximate)
print(f"\nRegular __dict__ size: {sys.getsizeof(regular.__dict__)} bytes")
print("Slotted has no __dict__, saves memory!")

In [ ]:
# ✏️ Exercise 3: Create a slotted class called Vector3D with:
# - Slots for x, y, z coordinates
# - Method magnitude() that returns the length of the vector
# Verify it doesn't have __dict__

# Your code here:


---
## 4. Descriptors

**Overview:** Descriptors control attribute access via `__get__`, `__set__`, `__delete__`.

**When to Use:**
- Reusable attribute validation
- Computed attributes
- Implementing properties at scale

**Workflow:**
```python
class Descriptor:
    def __get__(self, obj, objtype=None):
        return value
    
    def __set__(self, obj, value):
        # validate and store
        pass
```

In [ ]:
# Example: Descriptors
class PositiveNumber:
    """Descriptor that ensures value is positive"""
    
    def __set_name__(self, owner, name):
        self.name = name
        self.private_name = f'_{name}'
    
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.private_name, None)
    
    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name} must be positive, got {value}")
        setattr(obj, self.private_name, value)

class TypeChecked:
    """Descriptor that enforces type"""
    
    def __init__(self, expected_type):
        self.expected_type = expected_type
    
    def __set_name__(self, owner, name):
        self.name = name
        self.private_name = f'_{name}'
    
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.private_name, None)
    
    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name} must be {self.expected_type.__name__}")
        setattr(obj, self.private_name, value)

class Product:
    name = TypeChecked(str)
    price = PositiveNumber()
    quantity = PositiveNumber()
    
    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price
        self.quantity = quantity

# Test the descriptors
product = Product("Laptop", 999.99, 10)
print(f"Product: {product.name}, ${product.price}, qty: {product.quantity}")

# Validation in action
try:
    product.price = -100
except ValueError as e:
    print(f"Error: {e}")

try:
    product.name = 123
except TypeError as e:
    print(f"Error: {e}")

In [ ]:
# ✏️ Exercise 4: Create a descriptor called RangeValidator that:
# - Takes min_val and max_val in __init__
# - Validates that set values are within the range
# Use it in a Person class for age (0-150)

# Your code here:


---
## 5. Metaclasses

**Overview:** Metaclasses are "classes of classes" - they control class creation.

**When to Use:**
- Automatic registration of classes
- Enforcing coding standards
- Framework development

**Workflow:**
```python
class MyMeta(type):
    def __new__(mcs, name, bases, namespace):
        # Modify class before creation
        return super().__new__(mcs, name, bases, namespace)

class MyClass(metaclass=MyMeta):
    pass
```

In [ ]:
# Example: Metaclasses

# Example 1: Auto-registration metaclass
class PluginRegistry(type):
    """Metaclass that registers all plugin classes"""
    plugins = {}
    
    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        if name != 'Plugin':  # Don't register base class
            mcs.plugins[name] = cls
        return cls

class Plugin(metaclass=PluginRegistry):
    """Base class for plugins"""
    pass

class AuthPlugin(Plugin):
    def authenticate(self):
        return "Authenticating..."

class LoggingPlugin(Plugin):
    def log(self, msg):
        return f"LOG: {msg}"

print(f"Registered plugins: {PluginRegistry.plugins}")

# Example 2: Enforcing method naming convention
class EnforceNamingMeta(type):
    def __new__(mcs, name, bases, namespace):
        for attr_name in namespace:
            if callable(namespace[attr_name]) and not attr_name.startswith('_'):
                if not attr_name.islower():
                    raise ValueError(f"Method {attr_name} must be lowercase")
        return super().__new__(mcs, name, bases, namespace)

class GoodClass(metaclass=EnforceNamingMeta):
    def my_method(self):
        pass

# This would raise error:
# class BadClass(metaclass=EnforceNamingMeta):
#     def MyMethod(self):  # ValueError!
#         pass

print("GoodClass created successfully!")

In [ ]:
# ✏️ Exercise 5: Create a metaclass called SingletonMeta that:
# - Ensures only one instance of a class can exist
# - Returns existing instance if already created
# Test with a Database class

# Your code here:


---
## 6. Context Managers with Classes

**Overview:** Classes can be context managers by implementing `__enter__` and `__exit__`.

**When to Use:**
- Resource management (files, connections)
- Setup/teardown patterns
- Transaction handling

**Workflow:**
```python
class MyContext:
    def __enter__(self):
        # Setup code
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        # Cleanup code
        return False  # Re-raise exceptions
```

In [ ]:
# Example: Context Managers
import time

class Timer:
    """Context manager for timing code blocks"""
    
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time.perf_counter()
        self.elapsed = self.end - self.start
        print(f"Elapsed time: {self.elapsed:.4f} seconds")
        return False

class DatabaseConnection:
    """Simulated database connection context manager"""
    
    def __init__(self, host):
        self.host = host
        self.connected = False
    
    def __enter__(self):
        print(f"Connecting to {self.host}...")
        self.connected = True
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"Disconnecting from {self.host}...")
        self.connected = False
        if exc_type is not None:
            print(f"Error occurred: {exc_val}")
        return False  # Don't suppress exceptions
    
    def query(self, sql):
        if not self.connected:
            raise RuntimeError("Not connected!")
        return f"Executing: {sql}"

# Using Timer
with Timer() as t:
    sum(range(1000000))

# Using DatabaseConnection
with DatabaseConnection("localhost") as db:
    print(db.query("SELECT * FROM users"))

In [ ]:
# ✏️ Exercise 6: Create a context manager class called TempDirectory that:
# - Creates a temporary directory in __enter__
# - Returns the path to the directory
# - Deletes the directory and contents in __exit__
# (Hint: use tempfile and shutil modules)

# Your code here:


---
## 7. Mixins

**Overview:** Mixins are small classes that provide specific functionality to be mixed into other classes.

**When to Use:**
- Add reusable functionality
- Avoid deep inheritance hierarchies
- Composition over inheritance

**Workflow:**
```python
class SerializeMixin:
    def to_json(self):
        return json.dumps(self.__dict__)

class MyClass(SerializeMixin, BaseClass):
    pass
```

In [ ]:
# Example: Mixins
import json
from datetime import datetime

class JSONSerializerMixin:
    """Adds JSON serialization capability"""
    def to_json(self):
        return json.dumps(self.__dict__, default=str)
    
    @classmethod
    def from_json(cls, json_str):
        return cls(**json.loads(json_str))

class TimestampMixin:
    """Adds timestamp tracking"""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.created_at = datetime.now()
        self.updated_at = datetime.now()
    
    def touch(self):
        self.updated_at = datetime.now()

class ComparableMixin:
    """Adds comparison based on a key"""
    def _compare_key(self):
        raise NotImplementedError
    
    def __lt__(self, other):
        return self._compare_key() < other._compare_key()
    
    def __eq__(self, other):
        return self._compare_key() == other._compare_key()

# Using mixins
class User(JSONSerializerMixin, TimestampMixin, ComparableMixin):
    def __init__(self, name, email, age):
        self.name = name
        self.email = email
        self.age = age
        super().__init__()
    
    def _compare_key(self):
        return self.age

user1 = User("Alice", "alice@example.com", 30)
user2 = User("Bob", "bob@example.com", 25)

print(f"User as JSON: {user1.to_json()}")
print(f"\nuser1 < user2: {user1 < user2}")
print(f"Created at: {user1.created_at}")

In [ ]:
# ✏️ Exercise 7: Create mixins for a logging system:
# - LoggerMixin: adds log(message) method that prints with timestamp
# - ValidatableMixin: adds validate() method (returns True by default)
# Create an Order class using both mixins

# Your code here:


---
## 8. Method Resolution Order (MRO) Deep Dive

**Overview:** MRO determines the order in which base classes are searched when looking for a method.

**Key Concepts:**
- Python uses C3 Linearization algorithm
- `super()` follows MRO, not just parent
- Check with `ClassName.__mro__` or `ClassName.mro()`

In [ ]:
# Example: MRO Deep Dive

class A:
    def method(self):
        print("A.method")
        super().method() if hasattr(super(), 'method') else None

class B(A):
    def method(self):
        print("B.method")
        super().method()

class C(A):
    def method(self):
        print("C.method")
        super().method()

class D(B, C):
    def method(self):
        print("D.method")
        super().method()

print("MRO for class D:")
for i, cls in enumerate(D.__mro__):
    print(f"  {i}: {cls.__name__}")

print("\nCalling D().method():")
d = D()
d.method()

# Diamond problem demonstration
print("\n--- Diamond Problem ---")
print("Order: D -> B -> C -> A -> object")
print("super() in B calls C.method(), not A.method()!")

In [ ]:
# ✏️ Exercise 8: Create a diamond inheritance structure where:
# - Base class has greet() that prints "Base"
# - Left and Right inherit from Base, each adding to greeting
# - Child inherits from both Left and Right
# Use super() properly and print the MRO

# Your code here:


---
## 9. Class Decorators

**Overview:** Decorators can modify or enhance classes, not just functions.

**When to Use:**
- Add methods to classes
- Register classes
- Modify class behavior

**Workflow:**
```python
def my_decorator(cls):
    cls.new_attr = "added"
    return cls

@my_decorator
class MyClass:
    pass
```

In [ ]:
# Example: Class Decorators
import functools

def add_repr(cls):
    """Decorator that adds a nice __repr__ method"""
    def __repr__(self):
        attrs = ', '.join(f'{k}={v!r}' for k, v in self.__dict__.items())
        return f'{cls.__name__}({attrs})'
    cls.__repr__ = __repr__
    return cls

def singleton(cls):
    """Decorator that makes a class a singleton"""
    instances = {}
    
    @functools.wraps(cls)
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    
    return get_instance

def log_methods(cls):
    """Decorator that logs all method calls"""
    for name, method in cls.__dict__.items():
        if callable(method) and not name.startswith('_'):
            def make_logged(m, n):
                @functools.wraps(m)
                def logged(*args, **kwargs):
                    print(f"Calling {n}")
                    return m(*args, **kwargs)
                return logged
            setattr(cls, name, make_logged(method, name))
    return cls

@add_repr
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

@singleton
class Config:
    def __init__(self):
        self.settings = {}

@log_methods
class Calculator:
    def add(self, a, b):
        return a + b
    
    def multiply(self, a, b):
        return a * b

# Test decorators
p = Person("Alice", 30)
print(f"add_repr: {p}")

c1 = Config()
c2 = Config()
print(f"\nsingleton: c1 is c2 = {c1 is c2}")

calc = Calculator()
print(f"\nlog_methods:")
print(f"Result: {calc.add(2, 3)}")

In [ ]:
# ✏️ Exercise 9: Create a class decorator called 'validate_types' that:
# - Reads type hints from __init__
# - Wraps __init__ to validate argument types
# - Raises TypeError if wrong type is passed

# Your code here:


---
## 10. Singleton Pattern

**Overview:** Ensure only one instance of a class exists.

**Implementation Methods:**
1. Using `__new__`
2. Using metaclass
3. Using decorator
4. Using module (Python's natural singleton)

In [ ]:
# Example: Singleton Patterns

# Method 1: Using __new__
class SingletonNew:
    _instance = None
    
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

# Method 2: Using metaclass
class SingletonMeta(type):
    _instances = {}
    
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class DatabaseMeta(metaclass=SingletonMeta):
    def __init__(self):
        self.connection = "Connected"

# Test Method 1
s1 = SingletonNew()
s2 = SingletonNew()
print(f"__new__ singleton: s1 is s2 = {s1 is s2}")

# Test Method 2
db1 = DatabaseMeta()
db2 = DatabaseMeta()
print(f"Metaclass singleton: db1 is db2 = {db1 is db2}")

# Thread-safe singleton
import threading

class ThreadSafeSingleton:
    _instance = None
    _lock = threading.Lock()
    
    def __new__(cls):
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:
                    cls._instance = super().__new__(cls)
        return cls._instance

print(f"\nThread-safe singleton works!")

In [ ]:
# ✏️ Exercise 10: Create a thread-safe singleton Logger class that:
# - Has a log(message) method
# - Stores all messages in a list
# - Has get_logs() to retrieve all messages
# Verify singleton behavior in multiple calls

# Your code here:


---
## 🎯 Final Challenge: Plugin System

Build a plugin system that combines multiple advanced concepts:

1. Abstract base class `Plugin` with abstract methods
2. Metaclass for auto-registration
3. Dataclass for plugin metadata
4. Context manager for plugin lifecycle

In [ ]:
# Final Challenge: Plugin System
# Your code here:


---
## 📝 Quick Reference Summary

| Concept | Key Elements | Use Case |
|---------|-------------|----------|
| ABC | `@abstractmethod`, `ABC` | Enforce interface contracts |
| Dataclass | `@dataclass`, `field()` | Data containers, reduce boilerplate |
| `__slots__` | `__slots__ = [...]` | Memory optimization |
| Descriptors | `__get__`, `__set__` | Reusable attribute logic |
| Metaclass | `class Meta(type)` | Control class creation |
| Context Manager | `__enter__`, `__exit__` | Resource management |
| Mixins | Multiple inheritance | Add reusable functionality |
| MRO | `Class.__mro__` | Understand method lookup |
| Class Decorator | `@decorator` on class | Modify class behavior |
| Singleton | `__new__` or metaclass | Single instance pattern |